# 第一周：LLaVA 从单图推理到 30 张图基线

这是一份可以逐格运行、修改的教学笔记本。运行地点：**云端 Linux GPU 的 JupyterLab**。第一次先完成第 0–6 节，看到自己的图片对应的回答，再进行后面的评测。

路线：图片 → processor（图像预处理 + 文本分词）→ 视觉编码器 → projector → 语言模型 → generate → 解码。

本笔记本选择 Transformers **4.57.1** 的 API，采用单 GPU FP16 推理；这个版本选择不是已实测的环境锁定。PyTorch 使用云平台提供的 CUDA 版，实际兼容性和显存需求需要在你的实例验证。

官方参考：[版本对应的 LLaVA 文档](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/llava)、[模型卡](https://huggingface.co/llava-hf/llava-1.5-7b-hf)。模型会下载到云端缓存，图片在当前实例参与推理。

**运行方法**：选择 Python 内核，按 Shift+Enter 逐格执行。代码依赖前面的变量，报错时先修复当前格。不要在安装依赖后直接“Run All”，应先重启内核。未提供真实图片和标注，因此这里没有预填运行结果。


## 0. 查看云端环境
先运行这一格，确认内核的 Python 路径、GPU 和磁盘空间。nvidia-smi 中的 CUDA Version 是驱动支持能力；后面 torch.version.cuda 才是 PyTorch 的 CUDA runtime。若无 GPU，先检查实例和内核。


In [ ]:
import sys, subprocess
print("Python:", sys.version)
print("内核解释器:", sys.executable)
for command in (["nvidia-smi"], ["df", "-h"]):
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout or result.stderr)


## 1. 安装推理依赖（首次运行）
%pip 将依赖装到当前内核对应的环境。保留平台提供的 CUDA PyTorch；如果尚未安装 torch，请先按 [PyTorch 官方安装选择器](https://pytorch.org/get-started/locally/) 根据云端系统和驱动安装。

执行下面一格后，在 JupyterLab 菜单选择 **Kernel → Restart Kernel**，再从第 0 节运行；第二次跳过安装格。

不需要安装原生 LLaVA 仓库，也不需要先装训练用的 DeepSpeed。


In [ ]:
%pip install "transformers==4.57.1" "accelerate>=0.26,<2" "huggingface_hub>=0.34,<1" pillow sentencepiece protobuf safetensors


## 2. 配置路径和推理参数
如果笔记本放在仓库 src/ 内，代码会自动找到项目根目录；否则修改 PROJECT_ROOT 为你的云端仓库路径。

先把一张商品图上传到 data/images/001.jpg。只需修改 IMAGE_PATH 和 QUESTION 就可以换图、换问题。

max_new_tokens 限制新增回答长度；do_sample=False 使用贪心生成，便于比较。每次运行创建独立输出目录。


In [ ]:
import json, time, hashlib, inspect, importlib.metadata as metadata
from pathlib import Path
from datetime import datetime, timezone
import torch
import transformers
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
from huggingface_hub import model_info
from transformers import AutoProcessor, LlavaForConditionalGeneration

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
# 如果需要，改成：PROJECT_ROOT = Path("/你的云端路径/Mllm_learning")
IMAGE_PATH = PROJECT_ROOT / "data/images/001.jpg"
ANNOTATIONS_PATH = PROJECT_ROOT / "data/annotations.jsonl"
MODEL_ID = "llava-hf/llava-1.5-7b-hf"
REQUESTED_REVISION = "main"  # 首次解析成 commit；复现实验时换成保存的 commit
QUESTION = "What is shown in this image?"
GENERATION = {"max_new_tokens": 128, "do_sample": False, "num_beams": 1}
SEED = 42
DEVICE = "cuda:0"
DTYPE = torch.float16
torch.manual_seed(SEED)
assert torch.cuda.is_available(), "当前内核无法使用 CUDA，请先检查 GPU 和 PyTorch。"
assert transformers.__version__ == "4.57.1", "安装依赖后请重启内核，或使用与代码匹配的文档。"
torch.cuda.set_device(0)
RUN_DIR = PROJECT_ROOT / "outputs" / ("w1-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ"))
RUN_DIR.mkdir(parents=True, exist_ok=False)
def save_json(file_path, value):
    Path(file_path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
print("根目录:", PROJECT_ROOT)
print("图片存在:", IMAGE_PATH.is_file())
print("输出目录:", RUN_DIR)


## 3. 固定模型版本，保存环境
模型名不是版本：main 会变化。先取得实际 commit，再用同一个 commit 加载 processor 和模型。

环境文件和依赖清单将保存在本次 outputs/w1-…/ 中。下载失败时检查网络和模型缓存位置；不要在代码里写入访问 token。


In [ ]:
MODEL_REVISION = model_info(MODEL_ID, revision=REQUESTED_REVISION).sha
gpu = torch.cuda.get_device_properties(0)
try:
    repo_commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT,
                                 capture_output=True, text=True, check=True).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    repo_commit = None
environment = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version, "torch": torch.__version__,
    "transformers": transformers.__version__, "cuda_runtime": torch.version.cuda,
    "gpu": gpu.name, "gpu_total_gib": gpu.total_memory / 1024**3,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "repository_commit": repo_commit,
    "nvidia_smi": subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout,
}
save_json(RUN_DIR / "environment.json", environment)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
(RUN_DIR / "requirements-freeze.txt").write_text(freeze.stdout, encoding="utf-8")
save_json(RUN_DIR / "config.json", {
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "generation": GENERATION, "seed": SEED, "dtype": str(DTYPE),
    "device": DEVICE, "batch_size": 1, "question": QUESTION,
})
print(json.dumps(environment, ensure_ascii=False, indent=2))


## 4. 加载 processor 和模型（只加载一次）
processor 负责把图片和提示词转换成张量。模型包含视觉编码器、跨模态投影层和语言模型。

这份入门代码把完整模型放在一张 GPU 上，不做 CPU offload。FP16 权重的粗略量级为参数数量 × 2 字节，运行还需要视觉计算、KV cache 等空间；不能仅用权重大小判断显存是否够用。

若加载时出现 CUDA out of memory：重启内核释放旧模型，并记录实际显存；需要按 GPU 配置调整方案。不要反复加载模型。


In [ ]:
load_started = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=DTYPE,
    device_map={"": DEVICE}, low_cpu_mem_usage=True,
)
model.eval()
processor.tokenizer.padding_side = "left"
# 明确图像 token 展开规则，避免旧 processor 缺少属性。
processor.patch_size = model.config.vision_config.patch_size
processor.vision_feature_select_strategy = model.config.vision_feature_select_strategy
processor.num_additional_image_tokens = 1  # 此 checkpoint 的 CLIP 视觉编码器包含 CLS token
torch.cuda.synchronize()
load_info = {"load_seconds": time.perf_counter() - load_started,
             "allocated_gib_after_load": torch.cuda.memory_allocated() / 1024**3}
save_json(RUN_DIR / "model-load.json", load_info)
print(load_info)


## 5. 看清楚图片如何变成输入
先显示实际图片，避免路径正确但选错图。统一使用 RGB，并按 EXIF 修正方向。

聊天模板加入 USER、ASSISTANT 和图像占位符；processor 把文本变成 input_ids，把图片变成 pixel_values。整数 token ID 必须保留整数类型；只有浮点输入转换成 FP16。

观察 input_ids 的长度、pixel_values 的形状。形状以当前运行打印为准。


In [ ]:
def open_rgb(image_path):
    with Image.open(image_path) as source:
        return ImageOps.exif_transpose(source).convert("RGB")

def prepare_inputs(image, question):
    conversation = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": question}
    ]}]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt")
    inputs = {key: value.to(device=DEVICE, dtype=DTYPE if value.is_floating_point() else value.dtype)
              for key, value in inputs.items()}
    return prompt, inputs

image = open_rgb(IMAGE_PATH)
display(image)
prompt, inputs = prepare_inputs(image, QUESTION)
print("模板后的提示词:", prompt)
for name, value in inputs.items():
    print(name, tuple(value.shape), value.dtype, value.device)
print("图像 token 数:", (inputs["input_ids"] == model.config.image_token_id).sum().item())


## 6. 单图生成，再封装成可复用函数
generate 的输出包含输入提示词，所以按输入长度切掉前缀，只解码新生成的回答。inference_mode 关闭梯度记录，eval 关闭训练行为。

CUDA 运算是异步的；计时前后同步，才能测到真实完成时间。下面的耗时包括输入预处理、传到 GPU 和生成，不包括文件读取、模型加载及最后的文本解码。

峰值为 PyTorch 当前进程的 allocated/reserved 显存，不等于 nvidia-smi 的总占用。第一次生成可能包含初始化开销；本周保留原始耗时，不做性能排行。


In [ ]:
def infer_image(image, question):
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    prompt, inputs = prepare_inputs(image, question)
    with torch.inference_mode():
        output_ids = model.generate(**inputs, **GENERATION)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    new_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    answer = processor.batch_decode(new_ids, skip_special_tokens=True)[0].strip()
    return {
        "raw_answer": answer, "prompt": prompt, "seconds": elapsed,
        "new_tokens": new_ids.shape[1],
        "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
        "peak_reserved_gib": torch.cuda.max_memory_reserved() / 1024**3,
    }

# 第 5 节的 GPU 输入仅用于观察，释放后再计时。
del inputs
single_result = infer_image(image, QUESTION)
single_result["image"] = str(IMAGE_PATH)
single_result["image_sha256"] = hashlib.sha256(IMAGE_PATH.read_bytes()).hexdigest()
save_json(RUN_DIR / "single-image.json", single_result)
print(json.dumps(single_result, ensure_ascii=False, indent=2))


### 现在自己改两次
1. 修改 QUESTION，例如 “What color is the main product?”，重新运行第 5、6 节。
2. 修改 IMAGE_PATH 换一张图，重新运行第 5、6 节。

你应该能解释：为什么需要聊天模板？pixel_values 和 input_ids 分别来自哪里？为什么只解码新增 token？先理解这三点，再进入评测。

第 6 节重复执行会更新当前运行的单图文件；需要保留各次对比时另起一次 RUN_DIR 或另存文件名。


## 7. 准备人工标注的测试集
在 data/annotations.jsonl 中每行写一个 JSON 对象。图片路径相对项目根目录。下面只是**格式示例，不能直接当作真实标注**：

~~~json
{"id":"sample-001","image":"data/images/001.jpg","product_id":"product-001","split":"test","reference":{"category":"杯子","color":"白色","visible_text":"无法确定"}}
~~~

规则先定好，再让模型回答：category 为主要商品类别；color 为主要商品颜色，多色写“多色”；visible_text 抄录商品上的可见文字，无文字写“无”，有文字但无法辨认写“无法确定”。多段文字按从上到下、从左到右，以空格分隔。不明确的类别和颜色也写“无法确定”。

同一商品的不同照片使用相同 product_id。先收集 30 张，覆盖清晰、小字、遮挡、多物体。参考答案由你看图填写，不能把模型预测当作真值。若还没有 30 张，可先用少量真实标注验证流程。

评分采用首尾去空白后的精确匹配；第一轮不做同义词归一化。这样会把措辞差异也计错，必须在报告中说明。后续若改规则，应对所有样本一致应用并另存评测版本。


In [ ]:
FIELDS = ("category", "color", "visible_text")
UNKNOWN = "无法确定"
def validate_annotations(file_path):
    records, ids, products = [], set(), {}
    for line_number, line in enumerate(Path(file_path).read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        row = json.loads(line)
        for key in ("id", "image", "product_id", "split"):
            if not isinstance(row.get(key), str) or not row[key].strip():
                raise ValueError(f"第 {line_number} 行缺少有效 {key}")
        if row["id"] in ids:
            raise ValueError(f"重复 id: {row['id']}")
        ids.add(row["id"])
        previous_split = products.setdefault(row["product_id"], row["split"])
        if previous_split != row["split"]:
            raise ValueError(f"同一商品跨划分: {row['product_id']}")
        ref = row.get("reference")
        if not isinstance(ref, dict) or set(ref) != set(FIELDS):
            raise ValueError(f"第 {line_number} 行参考答案字段不完整")
        if not all(isinstance(ref[k], str) and ref[k].strip() for k in FIELDS):
            raise ValueError(f"第 {line_number} 行参考答案必须是非空字符串")
        if not (PROJECT_ROOT / row["image"]).is_file():
            raise FileNotFoundError(row["image"])
        if row["split"] == "test":
            records.append(row)
    if not records:
        raise ValueError("没有 test 样本")
    return records

records = validate_annotations(ANNOTATIONS_PATH)
dataset_info = {"annotation_sha256": hashlib.sha256(ANNOTATIONS_PATH.read_bytes()).hexdigest(),
                "test_samples": len(records),
                "image_sha256": {row["id"]: hashlib.sha256((PROJECT_ROOT / row["image"]).read_bytes()).hexdigest()
                                 for row in records}}
save_json(RUN_DIR / "dataset.json", dataset_info)
print("测试集样本数:", len(records))
if len(records) < 30:
    print("这是流程验证；第一周目标仍为 30 张真实标注图片。")


## 8. 固定属性提取提示词，严格解析 JSON
这时把自由问答换成三个固定字段。使用中文提示词和中文答案，可能受到这个模型中文能力的影响；先记录这个基线，不在测试集上反复调提示词。

这里只接受完整 JSON 对象及规定的三个字符串字段。代码围栏、解释文字、缺字段、重复字段都算格式失败，保留原始回答，不自动修补。JSON 合法不代表内容正确。


In [ ]:
ATTRIBUTE_PROMPT = (
    '请根据图片中主要商品，返回一个 JSON 对象，且只输出 JSON。'
    '必须包含三个非空字符串字段：category（商品类别）、color（主要颜色）、'
    'visible_text（商品上的可见文字）。类别和颜色用中文回答；'
    '多种主要颜色写“多色”。可见文字保持原文，多段文字按从上到下、从左到右，'
    '以空格分隔。没有文字写“无”；无法辨认或无法确定的字段写“无法确定”。'
    '不要推测品牌、材质或被遮挡的信息。'
)
def unique_object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError("重复 JSON 字段: " + key)
        result[key] = value
    return result

def parse_prediction(raw):
    try:
        value = json.loads(raw, object_pairs_hook=unique_object)
        if not isinstance(value, dict) or set(value) != set(FIELDS):
            raise ValueError("必须恰好包含三个约定字段")
        if not all(isinstance(value[k], str) and value[k].strip() for k in FIELDS):
            raise ValueError("字段必须为非空字符串")
        return {k: value[k].strip() for k in FIELDS}, None
    except (ValueError, TypeError) as error:
        return None, str(error)

save_json(RUN_DIR / "evaluation-config.json", {
    "prompt": ATTRIBUTE_PROMPT, "fields": FIELDS, "unknown_value": UNKNOWN,
    "scoring": "strict JSON schema; exact string match after strip; invalid prediction scores zero",
    "batch_size": 1, "generation": GENERATION,
})


## 9. 对测试集逐张推理
这是批量任务，但 GPU batch_size=1：复用同一个模型逐张处理，不把 30 张图同时塞进显存。每完成一张立即保存一行，失败也保存。

运行中断时已完成的行仍保留。再次运行本格会创建新的预测文件，不拼接到旧结果中。显存溢出将终止本轮，保留失败记录；修复环境后重新运行。


In [ ]:
PREDICTIONS_PATH = RUN_DIR / ("predictions-" + datetime.now(timezone.utc).strftime("%H%M%S.%f") + ".jsonl")
results = []
with PREDICTIONS_PATH.open("x", encoding="utf-8") as output_file:
    for index, row in enumerate(records, 1):
        result = {"id": row["id"], "image": row["image"], "product_id": row["product_id"],
                  "reference": row["reference"], "prediction": None, "valid_json": False,
                  "raw_answer": None, "error": None, "seconds": None}
        fatal_oom = False
        try:
            answer = infer_image(open_rgb(PROJECT_ROOT / row["image"]), ATTRIBUTE_PROMPT)
            result.update(answer)
            result["prediction"], result["error"] = parse_prediction(answer["raw_answer"])
            result["valid_json"] = result["prediction"] is not None
        except torch.cuda.OutOfMemoryError as error:
            result["error"] = "CUDA OOM: " + str(error)
            fatal_oom = True
        except Exception as error:
            result["error"] = type(error).__name__ + ": " + str(error)
        results.append(result)
        output_file.write(json.dumps(result, ensure_ascii=False) + "\n")
        output_file.flush()
        print(f"{index}/{len(records)} {row['id']} JSON有效={result['valid_json']}")
        if fatal_oom:
            raise RuntimeError("显存不足，本轮已停止，已写入失败记录。修复后重新运行。")
print("预测已保存:", PREDICTIONS_PATH)


## 10. 计算指标，生成待人工补充的报告
分母固定为全部 test 样本；解析失败的三个字段都计错。无法确定字段的编造比例，以所有参考标注为“无法确定”的字段为分母。

对格式失败或运行失败的样本，代码不能可靠判断其是否编造：另外报告这部分数量。编造比例只计可解析预测中的具体答案，遇到未解析项时只能视为下界，人工复核后再补充。

失败候选最多导出 10 条。失败少于 10 条时报告真实数量，不伪造案例；候选排序只是数据顺序，后续人工选择有代表性的案例。


In [ ]:
def score_results(results, total_samples):
    if len(results) != total_samples or not total_samples:
        raise ValueError("结果未覆盖全部样本，不能生成完整基线指标。")
    correct = {key: 0 for key in FIELDS}
    unknown_total = invented = unknown_unresolved = 0
    failures = []
    for row in results:
        pred = row["prediction"]
        wrong_fields = []
        for key in FIELDS:
            reference = row["reference"][key].strip()
            hit = pred is not None and pred[key] == reference
            correct[key] += int(hit)
            if not hit:
                wrong_fields.append(key)
            if reference == UNKNOWN:
                unknown_total += 1
                if pred is None:
                    unknown_unresolved += 1
                elif pred[key] != UNKNOWN:
                    invented += 1
        if wrong_fields:
            failures.append({**row, "wrong_fields": wrong_fields,
                             "manual_category": "待人工分类：感知/文字识别/指令遵循/格式/无法确定时编造",
                             "root_cause": "待核验"})
    valid = sum(row["valid_json"] for row in results)
    return {
        "samples": total_samples, "valid_json_count": valid,
        "json_valid_rate": valid / total_samples,
        "correct_fields": correct,
        "field_accuracy": sum(correct.values()) / (total_samples * len(FIELDS)),
        "per_field_accuracy": {key: correct[key] / total_samples for key in FIELDS},
        "unknown_fields_denominator": unknown_total,
        "invented_fields_count": invented,
        "unknown_unresolved_fields": unknown_unresolved,
        "invention_rate_lower_bound": invented / unknown_total if unknown_total else None,
        "inference_seconds_total": sum(row.get("seconds") or 0 for row in results),
        "peak_allocated_gib": max((row.get("peak_allocated_gib", 0) for row in results), default=0),
        "failed_samples": len(failures),
    }, failures[:10]

summary, failure_cases = score_results(results, len(records))
summary["predictions_file"] = PREDICTIONS_PATH.name
artifact_prefix = PREDICTIONS_PATH.stem
save_json(RUN_DIR / (artifact_prefix + "-metrics.json"), summary)
save_json(RUN_DIR / (artifact_prefix + "-failure-cases.json"), failure_cases)
report = (
    "# 第一周 LLaVA 基线报告\n\n"
    f"- 状态：推理与自动评分已完成，人工分析待补充\n"
    f"- 模型：{MODEL_ID} @ {MODEL_REVISION}\n"
    f"- GPU：{environment['gpu']}\n"
    f"- 测试集：{len(records)} 张，标注 SHA256：{dataset_info['annotation_sha256']}\n"
    f"- 预测文件：{PREDICTIONS_PATH.name}\n"
    "- 评分：严格 JSON 校验、字符串精确匹配，解析失败计错\n\n"
    "## 自动指标\n\n~~~json\n" + json.dumps(summary, ensure_ascii=False, indent=2) + "\n~~~\n\n"
    "## 人工失败分析\n\n请检查 failure-cases.json，填写每条的失败类别和证据。\n\n"
    "## 我的结论与局限\n\n待填写：内容错误与措辞差异、中文能力影响、未解析样本的编造情况。\n\n"
    "## 下一步\n\n待填写；不足 30 张时继续收集数据，保存最终基线后再比较提示词。\n"
)
(RUN_DIR / (artifact_prefix + "-report.md")).write_text(report, encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("输出位置:", RUN_DIR)


## 11. 入门源码观察：视觉特征和 projector
可选扩展，单图跑通后再执行。直接调用当前模型的视觉编码器并观察投影，不注册持久 hook。

CLIP 输出含 CLS token；此 checkpoint 使用 default 策略，去掉第一个 token 后，把每个 patch 的视觉向量经 projector 转成语言模型的 hidden size。这些特征随后替换序列中的图像 token 位置。

下面打印当前安装版本的函数文件位置。下一周再沿着 forward 跟踪多模态序列，不把 HF 实现与原生 LLaVA 的函数路径混为一谈。


In [ ]:
_, trace_inputs = prepare_inputs(image, QUESTION)
with torch.inference_mode():
    vision_outputs = model.model.vision_tower(
        trace_inputs["pixel_values"], output_hidden_states=True)
    visual = vision_outputs.hidden_states[model.config.vision_feature_layer]
    if model.config.vision_feature_select_strategy == "default":
        visual = visual[:, 1:, :]
    projected = model.model.multi_modal_projector(visual)
trace = {
    "pixels": list(trace_inputs["pixel_values"].shape),
    "selected_visual_features": list(visual.shape),
    "projected_features": list(projected.shape),
    "language_hidden_size": model.config.text_config.hidden_size,
    "source_file": inspect.getfile(type(model)),
    "transformers_version": transformers.__version__,
    "model_revision": MODEL_REVISION,
}
save_json(RUN_DIR / "tensor-trace.json", trace)
print(json.dumps(trace, ensure_ascii=False, indent=2))
print("HF 图像特征函数源码：")
print(inspect.getsource(type(model.model).get_image_features))
del trace_inputs, vision_outputs, visual, projected


## 12. 做一次受控改动：遮挡图中一个区域
修改 BOX，让它覆盖商品上的文字或一个关键区域。只改图片，不改问题或生成设置。原图不会被覆盖。

默认矩形只是演示位置，必须自己选有意义的区域。模型回答是否变化需要实际观察，不能先写结论。


In [ ]:
BOX = (0, 0, image.width // 2, image.height // 2)  # (左, 上, 右, 下)，请按实际图调整
occluded = image.copy()
ImageDraw.Draw(occluded).rectangle(BOX, fill="gray")
display(occluded)
before = infer_image(image, QUESTION)
after = infer_image(occluded, QUESTION)
comparison = {"question": QUESTION, "box": BOX, "before": before, "after": after}
save_json(RUN_DIR / "occlusion-comparison.json", comparison)
occluded.save(RUN_DIR / "occluded.png")
print("原图回答:", before["raw_answer"])
print("遮挡后回答:", after["raw_answer"])


## 完成后保存什么
- 输出目录：环境、依赖、commit、提示词、数据校验、原始预测、指标、失败候选和报告。
- 人工填写报告，把精简报告复制到 experiments/，更新 index.md 的实际进展。
- 云端实例释放前，将 data/、模型缓存和 outputs/ 备份到持久存储；这些目录被 Git 忽略，git push 不会保存它们。
- 提交这份笔记本时清空输出，避免把真实图片或长日志放入 Git。

常见卡点：找不到图片 → 看 PROJECT_ROOT 和 IMAGE_PATH；找不到 CUDA → 看内核和 CUDA 版 PyTorch；安装后版本未变化 → 重启内核；加载 OOM → 检查单 GPU 显存并调整推理方案；JSON 失败 → 保留原文作为基线；标注文件缺失 → 先完成第 7 节人工标注。
